# Belousov–Zhabotinsky oscillator (Oregonator)

*chemistry · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/chemistry/belousov_zhabotinsky)

A stirred beaker of bromate, malonic acid and a cerium catalyst flips between colours like a clock. The scaled two-variable Oregonator is a stiff ODE; a positive production–loss (Patankar) step integrates a whole batch of stoichiometric factors $f$ at once, the trace of the Jacobian locates the two Hopf bifurcations that bound the oscillations, and the same kinetics with diffusion turns a broken wave into a rotating spiral.

**Vector form**

$$\varepsilon\,\dot x = x(1 - x) - f\,z\,\frac{x - q}{x + q},\qquad \dot z = x - z,\qquad \text{Hopf: } \operatorname{tr} J(x^*, z^*) = 0,\ \det J > 0$$

**Index (tensor) form**

$$x_b \leftarrow \frac{x_b + \Delta t\,P_b/\varepsilon}{1 + \Delta t\,L_b/\varepsilon},\quad P_b = x_b + \frac{f_b z_b q}{x_b + q},\quad L_b = x_b + \frac{f_b z_b}{x_b + q} \qquad z_b \leftarrow \frac{z_b + \Delta t\,x_b}{1 + \Delta t} \qquad \operatorname{tr} J_b = J_{b\alpha\alpha}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $f_b$ | `f_values(n, device)` | (B,) | stoichiometric factor (bromide made per catalyst reduced) of reaction $b$, from $0.3$ to $3$ |
| $x_b,\ z_b$ | `x, z` | (B,) | scaled $[\mathrm{HBrO_2}]$ (autocatalyst) and $[\mathrm{Ce^{4+}}]$ (oxidised catalyst) of every reaction |
| $P_b,\ L_b$ | `production_loss(x, z, f, q)` | (B,) | non-negative gain and loss rate with $\varepsilon\dot x = P - Lx$ |
| $x_b \leftarrow \dots$ | `step(x, z, f, eps, q, dt)` | (B,) | Patankar–Euler: losses implicit, gains explicit, so $x$ stays positive however stiff |
| $x^* = z^*$ | `steady_state(f, q)` | (B,) | positive root of $x^2 + (f + q - 1)x - q(1 + f) = 0$ |
| $J_{b\alpha\beta}$ | `jacobian(x, z, f, eps, q)` | (B, 2, 2) | linearised kinetics $\partial \dot y_\alpha / \partial y_\beta$ at the steady state |
| $J_{b\alpha\alpha}$ | `einsum('bii->b', J)` | (B,) | repeated $\alpha$ ⇒ summed: the trace; its sign change marks a Hopf bifurcation |
| $z = \frac{x(1-x)(x+q)}{f(x-q)}$ | `x_nullcline(x, f, q)` | (X,) | the N-shaped $\dot x = 0$ curve of the phase portrait |
| $\nabla^2 x_{ij}$ | `laplacian(x, h)` | (n, n) | 5-point stencil with no-flux edges; only $x$ diffuses in the 2D medium |

The Belousov–Zhabotinsky reaction (discovered by Belousov in the 1950s, explained by Field, Kőrös and Noyes in 1972) oxidises malonic acid with bromate. Bromide ions inhibit the autocatalytic production of $\mathrm{HBrO_2}$; while bromide is present the reaction creeps, once it is used up $\mathrm{HBrO_2}$ explodes and oxidises the catalyst (the colour change), and the oxidised catalyst then makes new bromide, which shuts the burst down. The Oregonator condenses this into a few steps and, after scaling and eliminating the fast bromide, into two equations. The constants $\varepsilon \approx 0.04$ and $q \approx 8\times10^{-4}$ are approximate values from fits of the Field–Kőrös–Noyes rate constants; recipes differ and so do published values. Because $\varepsilon$ and $q$ are small the $x$ equation is stiff: $x$ jumps by a factor of about a thousand in a time of order $\varepsilon$, then crawls along a branch of its nullcline. Plain RK4 needs steps of about $5\times10^{-4}$ to stay stable; writing $\varepsilon\dot x$ as production minus loss and taking the loss implicitly keeps $x$ positive at any step, and $\Delta t = 2\times10^{-3}$ gets the period right to better than 1 %. The steady state (the crossing of the two nullclines) is stable when it sits on an outer branch of the N-shaped $\dot x = 0$ curve and unstable on the middle branch, where a limit cycle surrounds it; the trace of $J$ changes sign at the two Hopf points, near $f = 0.5$ and $f = 2.4$ for small $q$. Try a larger $\varepsilon$: the window shrinks and finally closes. In an unstirred thin layer the catalyst is held in place while $\mathrm{HBrO_2}$ diffuses, so a wave of oxidation runs through the dish; a broken wave front curls at its free end into the famous rotating spiral, whose rotation period is shorter than the bulk oscillation, so it takes over the whole medium.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/chemistry/belousov_zhabotinsky/main.py
"""Main loop: batch of f → stability of the steady state (Hopf points) → stiff integration → 2D spiral → GIF."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import jacobian, steady_state, trace


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    eps, q, dt = params["eps"], tensors.Q, tensors.DT
    t_max = 2.0 if fast else params["t_max"]
    n_steps = round(t_max / dt)
    f = tensors.f_values(12 if fast else 55, dev)  # (B,)
    f_sel = params["f"]
    print(f"Oregonator, ε = {eps}, q = {q} (approximate), Δt = {dt}, {n_steps} Patankar–Euler steps, {len(f)} f values")

    # --- linear stability: where does the steady state start to oscillate? ---
    f_fine = torch.linspace(0.05, 4.0, 4000, dtype=torch.float64)
    hopf = solve.hopf_points(f_fine, eps, q)
    print("Hopf bifurcations (tr J = 0) at f = " + ", ".join(f"{h:.3f}" for h in hopf.tolist()))
    fs = torch.tensor([f_sel], dtype=torch.float64)
    xs = steady_state(fs, q)
    J = jacobian(xs, xs, fs, eps, q)[0]
    lam = torch.linalg.eigvals(J)
    print(
        f"f = {f_sel}: steady state x* = z* = {xs.item():.5f}, tr J = {trace(J[None]).item():+.2f}, "
        f"det J = {torch.linalg.det(J).item():.2f}, eigenvalues {lam[0].item():.3f}, {lam[1].item():.3f}"
    )

    # --- the batch of well-mixed reactions, plus the selected f ---
    f_all = torch.cat([f, torch.tensor([f_sel], dtype=torch.float64, device=dev)])
    x0, z0 = tensors.initial_state(f_all)
    threads = torch.get_num_threads()
    torch.set_num_threads(1)  # thousands of tiny steps: one thread is far steadier than many on a busy CPU
    try:
        t, x, z = solve.integrate(x0, z0, f_all, eps, q, dt, n_steps)
        spiral = None
        if params["spiral"] == "on":
            n = 48 if fast else params["grid"]
            sx, sz = tensors.broken_wave(n, f_sel, dev)
            spiral = solve.spiral(sx, sz, f_sel, eps, q, dt, tensors.H, n_steps)
    finally:
        torch.set_num_threads(threads)
    lo, hi = solve.amplitude(x)
    osc = hi / lo > 1.5
    if osc[:-1].any():
        fo = f[osc[:-1]]
        print(f"simulation: x oscillates for f in [{fo.min():.3f}, {fo.max():.3f}] (grid step {f[1] - f[0]:.3f})")
    else:
        print("simulation: no oscillations at this ε")
    T = solve.period(t, x[:, -1])
    print(f"f = {f_sel}: x swings between {lo[-1]:.2e} and {hi[-1]:.3f}" + (f", period {T:.2f}" if T else ""))

    # 2D medium (integrated above): a broken wave curls into a spiral
    if spiral is not None:
        n = spiral.shape[-1]
        print(f"spiral: {n}×{n} grid, h = {tensors.H}, D_x = 1, z ∈ [{spiral[-1].min():.3f}, {spiral[-1].max():.3f}]")
    return plot.animate(t, x, z, f_all, lo, hi, hopf, eps, q, spiral, out_dir / "belousov_zhabotinsky.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/chemistry/belousov_zhabotinsky/equation.py
r"""Physics → tensor translation: the two-variable Oregonator model of the Belousov–Zhabotinsky reaction.

Indices: b value of the stoichiometric factor f_b in the batch, (i, j) grid cell of the 2D medium,
α, β ∈ {x, z} the two variables of the Jacobian.

Scaled Oregonator (Field–Kőrös–Noyes mechanism, reduced to two variables by Tyson and Fife), with
x ∝ [HBrO₂] (the autocatalyst), z ∝ [Ce⁴⁺] (the oxidised catalyst); bromide is fast and has been eliminated:
    ε dx/dt = x (1 - x) - f z (x - q) / (x + q)
      dz/dt = x - z
The x equation is stiff (ε ≪ 1 and q ≪ 1). Written as production minus loss, ε dx/dt = P - L x with
    P = x + f z q / (x + q) ≥ 0,     L = x + f z / (x + q) ≥ 0,
the Patankar–Euler step x ← (x + Δt P/ε) / (1 + Δt L/ε) stays positive and stable for any Δt.

Linear stability of the steady state (x*, z* = x*): eigenvalues of the Jacobian J_αβ = ∂(dy_α/dt)/∂y_β.
Oscillations start (Hopf bifurcation) where the trace J_αα (α summed) crosses zero while det J > 0.
"""

import torch


def production_loss(x: torch.Tensor, z: torch.Tensor, f: torch.Tensor, q: float):
    """P and L of ε dx/dt = P - L x, both non-negative.   Same shape as x."""
    P = x + f * z * q / (x + q)
    L = x + f * z / (x + q)
    return P, L


def rhs(x: torch.Tensor, z: torch.Tensor, f: torch.Tensor, eps: float, q: float):
    """dx/dt = [x(1 - x) - f z (x - q)/(x + q)] / ε,   dz/dt = x - z.   Same shape as x."""
    return (x * (1 - x) - f * z * (x - q) / (x + q)) / eps, x - z


def steady_state(f: torch.Tensor, q: float) -> torch.Tensor:
    """x* = z* solves (1 - x)(x + q) = f (x - q):  x² + (f + q - 1) x - q (1 + f) = 0, positive root."""
    a = 1 - f - q
    return (a + torch.sqrt(a * a + 4 * q * (1 + f))) / 2


def jacobian(x: torch.Tensor, z: torch.Tensor, f: torch.Tensor, eps: float, q: float) -> torch.Tensor:
    """J_bαβ = ∂(ẋ, ż)/∂(x, z) at (x_b, z_b).   (B, 2, 2)"""
    Jxx = (1 - 2 * x - 2 * f * q * z / (x + q) ** 2) / eps
    Jxz = -f * (x - q) / (x + q) / eps
    one = torch.ones_like(x)
    return torch.stack([torch.stack([Jxx, Jxz], -1), torch.stack([one, -one], -1)], -2)


def trace(J: torch.Tensor) -> torch.Tensor:
    """tr J_b = J_bαα, α summed: the sum of the two eigenvalues; > 0 (with det > 0) means growing oscillations.  (B,)"""
    return torch.einsum("bii->b", J)


def x_nullcline(x: torch.Tensor, f: float, q: float) -> torch.Tensor:
    """The curve where dx/dt = 0: z = x (1 - x)(x + q) / (f (x - q)), for q < x < 1."""
    return x * (1 - x) * (x + q) / (f * (x - q))


def laplacian(u: torch.Tensor, h: float) -> torch.Tensor:
    """5-point Laplacian of u (n, n) with no-flux (Neumann) edges: a cell on the edge is its own missing neighbour."""
    up = torch.cat([u[:1], u[:-1]], 0)  # u_{i-1, j}
    down = torch.cat([u[1:], u[-1:]], 0)  # u_{i+1, j}
    left = torch.cat([u[:, :1], u[:, :-1]], 1)  # u_{i, j-1}
    right = torch.cat([u[:, 1:], u[:, -1:]], 1)  # u_{i, j+1}
    return (up + down + left + right - 4 * u) / h**2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/chemistry/belousov_zhabotinsky/tensors.py
"""Build the concrete tensors: the batch of f values, the starting points and the broken wave that seeds a spiral.

The scaled Oregonator has two small constants, ε (set from the page, default 0.04) and q below. Both are ratios of
Field–Kőrös–Noyes rate constants and reactant concentrations; the values used here (Tyson 1985) are approximate, and
published fits range over ε ≈ 0.01–0.1 and q ≈ 2×10⁻⁴–8×10⁻⁴ depending on the recipe.
"""

import torch

from .equation import steady_state

Q = 8e-4  # q: a small ratio of rate constants (approximate)
DT = 2e-3  # time step of the Patankar–Euler scheme, in units of the z relaxation time
H = 0.5  # grid spacing of the 2D medium, in units where D_x = 1


def f_values(n: int, device: torch.device) -> torch.Tensor:
    """A batch of stoichiometric factors f_b spanning both Hopf bifurcations.   (B,)"""
    return torch.linspace(0.3, 3.0, n, dtype=torch.float64, device=device)


def initial_state(f: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Start every member of the batch 50 % above its steady state in x.   (B,), (B,)"""
    xs = steady_state(f, Q)
    return 1.5 * xs, xs.clone()


def broken_wave(n: int, f: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """x, z on an n × n grid (n, n): rest everywhere, an excited band (high x) along half of the middle row,
    and a refractory band (high z) just above it, so the wave can only travel down and its free end curls up."""
    xs = steady_state(torch.tensor(f, dtype=torch.float64), Q).item()
    x = torch.full((n, n), xs, dtype=torch.float64, device=device)
    z = torch.full((n, n), xs, dtype=torch.float64, device=device)
    w = max(2, n // 20)
    x[n // 2 : n // 2 + w, : n // 2] = 0.8
    z[n // 2 - 2 * w : n // 2, : n // 2] = 0.3
    return x, z


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/chemistry/belousov_zhabotinsky/solve.py
"""Positive, stable Patankar–Euler time stepping of the stiff Oregonator: a batch of f values, and a 2D medium."""

import torch

from problems._common.progress import steps

from .equation import jacobian, laplacian, production_loss, steady_state, trace


def step(x, z, f, eps: float, q: float, dt: float, diffusion: torch.Tensor | float = 0.0):
    """One step: x ← (x + Δt (P/ε + D∇²x)) / (1 + Δt L/ε), then z ← (z + Δt x) / (1 + Δt).

    Every loss term is taken at the new time (implicit) and every gain at the old one, so the stiff x relaxes
    without overshooting below zero. The z step is backward Euler, exact for its linear decay."""
    P, L = production_loss(x, z, f, q)
    x = ((x + dt * (P / eps + diffusion)) / (1 + dt * L / eps)).clamp(min=1e-12)  # clamp guards the D∇²x < 0 corner
    z = (z + dt * x) / (1 + dt)
    return x, z


def integrate(x0, z0, f, eps: float, q: float, dt: float, n_steps: int, n_frames: int = 300):
    """Batch of well-mixed reactions. Returns times (T,), x (T, B), z (T, B) at n_frames evenly spaced times."""
    every = max(1, n_steps // n_frames)
    x, z = x0.clone(), z0.clone()
    t, xs, zs = [0.0], [x], [z]
    for k in steps(n_steps, "Oregonator steps"):
        x, z = step(x, z, f, eps, q, dt)
        if (k + 1) % every == 0:
            t.append((k + 1) * dt)
            xs.append(x)
            zs.append(z)
    return torch.tensor(t), torch.stack(xs), torch.stack(zs)


def spiral(x0, z0, f: float, eps: float, q: float, dt: float, h: float, n_steps: int, n_frames: int = 90):
    """2D medium where only x diffuses (D_x = 1; the catalyst z is held in a gel). Returns z snapshots (T, n, n)."""
    every = max(1, n_steps // n_frames)
    x, z = x0.clone(), z0.clone()
    frames = [z.clone()]
    for k in steps(n_steps, "spiral-wave steps"):
        x, z = step(x, z, f, eps, q, dt, diffusion=laplacian(x, h))
        if (k + 1) % every == 0:
            frames.append(z.clone())
    return torch.stack(frames)


def hopf_points(f: torch.Tensor, eps: float, q: float) -> torch.Tensor:
    """Values of f where tr J at the steady state changes sign (det J > 0 there): the Hopf bifurcations.

    f (F,) a fine grid → the crossings, linearly interpolated."""
    xs = steady_state(f, q)
    tr = trace(jacobian(xs, xs, f, eps, q))  # (F,)
    i = torch.nonzero(torch.sign(tr[1:]) != torch.sign(tr[:-1])).flatten()
    return f[i] - tr[i] * (f[i + 1] - f[i]) / (tr[i + 1] - tr[i])


def period(t: torch.Tensor, x: torch.Tensor) -> float | None:
    """Mean time between upward crossings of the geometric mean of x's range, in the second half of the run."""
    half = len(t) // 2
    t, x = t[half:], x[half:].log()
    level = (x.min() + x.max()) / 2
    up = torch.nonzero((x[1:] > level) & (x[:-1] <= level)).flatten()
    if x.max() - x.min() < 0.5 or len(up) < 2:
        return None
    return (t[up[-1]] - t[up[0]]).item() / (len(up) - 1)


def amplitude(x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Minimum and maximum of x over the second half of the run (after the transient).   x (T, B) → (B,), (B,)"""
    late = x[x.shape[0] // 2 :]
    return late.min(0).values, late.max(0).values


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/chemistry/belousov_zhabotinsky/plot.py
"""Animate the oscillation: time series, phase portrait with nullclines, Hopf bifurcation diagram and spiral wave."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .equation import jacobian, rhs, steady_state, trace, x_nullcline

BLUE, RED, GREY, AMBER, DARK = "#2563eb", "#dc2626", "#9ca3af", "#f59e0b", "#374151"
LOG_TICKS = [1e-3, 1e-2, 1e-1, 1]
LOG_LABELS = ["0.001", "0.01", "0.1", "1"]  # plain labels: no mathtext and no minor ticks to redraw every frame


def _log_axis(ax, which: str = "y"):
    if which == "y":
        ax.set_yscale("log")
        ax.set_yticks(LOG_TICKS, LOG_LABELS)
    else:
        ax.set_xscale("log")
        ax.set_xticks(LOG_TICKS, LOG_LABELS)
    ax.minorticks_off()


def animate(t, x, z, f, lo, hi, hopf, eps: float, q: float, spiral, path: Path) -> Path:
    """t (T,); x, z (T, B + 1) with the selected f last; f (B + 1,); lo, hi (B + 1,); spiral (S, n, n) or None."""
    ts, X, Z = t.numpy(), x.cpu().numpy(), z.cpu().numpy()
    fv = f.cpu().numpy()
    f_sel = float(fv[-1])
    lo, hi = lo.cpu().numpy(), hi.cpu().numpy()

    fig, axs = plt.subplots(2, 2, figsize=(8.6, 6.0))
    fig.subplots_adjust(left=0.08, right=0.97, top=0.94, bottom=0.08, hspace=0.38, wspace=0.28)
    ax_t, ax_p, ax_b, ax_s = axs.flat

    # --- time series of x for the chosen f and two stable neighbours ---
    others = [int(np.abs(fv[:-1] - v).argmin()) for v in (0.4, 2.8)]
    for b in others:
        ax_t.plot(ts, X[:, b], color=GREY, lw=1)
        ax_t.text(ts[-1], X[-1, b], f" f={fv[b]:.2f}", fontsize=7, color="#6b7280", va="center", ha="right")
    ax_t.plot(ts, X[:, -1], color=BLUE, lw=1.4, label=f"x ∝ [HBrO₂], f = {f_sel:g}")
    ax_t.plot(ts, Z[:, -1], color=RED, lw=1.2, label="z ∝ [Ce⁴⁺]")
    _log_axis(ax_t)
    ax_t.set_ylim(3e-4, 1.5)
    ax_t.set_xlim(0, ts[-1])
    ax_t.set_xlabel("scaled time t", labelpad=1)
    ax_t.set_title("well-mixed oscillations (stiff: x jumps ×1000)", fontsize=9)
    ax_t.legend(fontsize=7, loc="lower right", frameon=False)
    (cursor,) = ax_t.plot([0, 0], [3e-4, 1.5], color=AMBER, lw=1)

    # --- phase portrait in (log x, z) with the nullclines ---
    xg = np.logspace(np.log10(q) + 0.002, 0, 400)
    ax_p.plot(xg, x_nullcline(torch.tensor(xg), f_sel, q).numpy(), color=DARK, lw=1.2, label="ẋ = 0")
    ax_p.plot(xg, xg, color=DARK, lw=1.2, ls=(0, (4, 3)), label="ż = 0  (z = x)")
    gx, gz = np.meshgrid(np.logspace(-3.3, 0, 15), np.linspace(0.0, 0.4, 13))
    dx, dz = rhs(torch.tensor(gx), torch.tensor(gz), torch.tensor(f_sel), eps, q)
    u = (dx / torch.tensor(gx)).numpy() * np.log10(np.e) / 3.3  # direction in the plotted (log x, z) coordinates
    v = dz.numpy() / 0.4
    norm = np.hypot(u, v) + 1e-12
    ax_p.quiver(gx, gz, u / norm, v / norm, color="#d1d5db", angles="xy", scale=30, width=0.004)
    late = len(ts) // 3
    ax_p.plot(X[late:, -1], Z[late:, -1], color=BLUE, lw=1.2)
    xs = steady_state(torch.tensor([f_sel], dtype=torch.float64), q).item()
    ax_p.plot(xs, xs, "o", mfc="white", mec=DARK, ms=6)
    (dot,) = ax_p.plot([], [], "o", color=AMBER, ms=7, mec=DARK)
    _log_axis(ax_p, "x")
    ax_p.set_xlim(5e-4, 1)
    ax_p.set_ylim(0, 0.4)
    ax_p.set_xlabel("x", labelpad=1)
    ax_p.set_ylabel("z", labelpad=1)
    ax_p.set_title(f"phase portrait, f = {f_sel:g} (open dot: steady state)", fontsize=9)
    ax_p.legend(fontsize=7, loc="upper right", frameon=False)

    # --- Hopf bifurcation diagram over the batch ---
    fine = torch.linspace(0.3, 3.0, 300, dtype=torch.float64)
    xf = steady_state(fine, q)
    stable = (trace(jacobian(xf, xf, fine, eps, q)) < 0).numpy()
    fn, xn = fine.numpy(), xf.numpy()
    ax_b.plot(fn, np.where(stable, xn, np.nan), color=DARK, lw=1.5, label="steady state (stable)")
    ax_b.plot(fn, np.where(stable, np.nan, xn), color=DARK, lw=1.2, ls=(0, (4, 3)), label="steady state (unstable)")
    osc = hi[:-1] / lo[:-1] > 1.5
    ax_b.plot(fv[:-1][osc], hi[:-1][osc], "o", color=BLUE, ms=3.5, label="limit cycle: max, min of x")
    ax_b.plot(fv[:-1][osc], lo[:-1][osc], "o", color=BLUE, ms=3.5)
    for h in hopf.tolist():
        ax_b.axvline(h, color=AMBER, lw=1.2)
        ax_b.text(h, 1.3, f" Hopf\n f={h:.2f}", fontsize=7, color="#b45309", va="top")
    ax_b.axvline(f_sel, color=GREY, lw=0.8, ls=(0, (2, 2)))
    _log_axis(ax_b)
    ax_b.set_ylim(3e-4, 1.5)
    ax_b.set_xlim(0.3, 3.0)
    ax_b.set_xlabel("stoichiometric factor f", labelpad=1)
    ax_b.set_ylabel("x", labelpad=1)
    ax_b.set_title(f"bifurcation diagram, {len(fv) - 1} f values in one batch", fontsize=9)
    ax_b.legend(fontsize=6.5, loc="center right", frameon=False)

    # --- 2D spiral wave ---
    ax_s.set_xticks([])
    ax_s.set_yticks([])
    ax_s.grid(False)
    image = None
    if spiral is not None:
        S = spiral.cpu().numpy()
        cmap = plt.get_cmap("RdBu_r", 12)  # oxidised catalyst red, reduced blue, like ferroin in a dish
        image = ax_s.imshow(S[0], cmap=cmap, vmin=0, vmax=max(0.05, float(S.max())), interpolation="nearest")
        title = ax_s.set_title("", fontsize=9)
    else:
        ax_s.text(0.5, 0.5, "2D spiral switched off", ha="center", va="center", transform=ax_s.transAxes)

    def update(i: int):
        cursor.set_data([ts[i], ts[i]], [3e-4, 1.5])
        dot.set_data([X[i, -1]], [Z[i, -1]])
        if image is not None:
            s = round(i / (len(ts) - 1) * (len(S) - 1))
            image.set_data(S[s])
            title.set_text(f"spiral wave in a dish, z at t = {ts[i]:.1f}")

    return save_gif(fig, update, frame_indices(len(ts), 50), path, fps=8)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'f': 1.0,
    'eps': 0.04,
    't_max': 30.0,
    'spiral': 'on',
    'grid': 128
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.chemistry.belousov_zhabotinsky")]:
    del sys.modules[name]
main = importlib.import_module("problems.chemistry.belousov_zhabotinsky.main")
problem_dir = Path("problems/chemistry/belousov_zhabotinsky")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
